# 05_06 - Stacking and Calibration

Run stacking meta-models and bounded calibration over saved Phase 5 probabilities.


## Contract

Reads base probabilities. Writes stacker metrics, selected calibrated probabilities and metadata.


In [1]:
# try/except: khối xử lý ngoại lệ
try:
    # import google.colab  # type: ignore: import thư viện google
    import google.colab  # type: ignore
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = True
# except: xử lý ngoại lệ — except ImportError:
except ImportError:
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = False

# if: điều kiện — if not IN_COLAB:
if not IN_COLAB:
    # raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 tr...: ném lỗi và dừng cell
    raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 training locally.")


In [2]:
# from google.colab import drive: import thư viện google
from google.colab import drive
# drive.mount("/content/drive"): mount Google Drive trên Colab
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
# import gc: giải phóng bộ nhớ
import gc
# import importlib: import thư viện importlib
import importlib
# import json: đọc/ghi JSON metadata
import json
# import os: biến môi trường hệ thống
import os
# import random: cố định seed ngẫu nhiên
import random
# import subprocess: chạy lệnh pip/cài package
import subprocess
# import sys: tham số Python runtime
import sys
# import time: đo thời gian thực thi
import time
# from datetime import datetime, timezone: import thư viện datetime
from datetime import datetime, timezone
# from itertools import combinations, product: import thư viện itertools
from itertools import combinations, product
# from pathlib import Path: quản lý đường dẫn
from pathlib import Path

# import joblib: lưu/tải object đã fit
import joblib
# import numpy: tính toán mảng số
import numpy as np
# import pandas: xử lý DataFrame
import pandas as pd
# from sklearn.metrics import (: thư viện machine learning scikit-learn
from sklearn.metrics import (
    # accuracy_score,: thực thi lệnh Python
    accuracy_score,
    # average_precision_score,: thực thi lệnh Python
    average_precision_score,
    # brier_score_loss,: thực thi lệnh Python
    brier_score_loss,
    # confusion_matrix,: thực thi lệnh Python
    confusion_matrix,
    # f1_score,: thực thi lệnh Python
    f1_score,
    # precision_recall_fscore_support,: thực thi lệnh Python
    precision_recall_fscore_support,
    # roc_auc_score,: thực thi lệnh Python
    roc_auc_score,
# ): đóng ngoặc gọi hàm
)

# SEED: biến cấu hình/hằng số của notebook
SEED = 42
# FAKE_LABEL: biến cấu hình/hằng số của notebook
FAKE_LABEL = 1
# REAL_LABEL: biến cấu hình/hằng số của notebook
REAL_LABEL = 0
# DEFAULT_THRESHOLD: biến cấu hình/hằng số của notebook
DEFAULT_THRESHOLD = 0.50
# TARGET_PRECISION_FAKE: biến cấu hình/hằng số của notebook
TARGET_PRECISION_FAKE = 0.975

# PROJECT_ROOT: biến cấu hình/hằng số của notebook
PROJECT_ROOT = Path(os.environ.get("FAKE_REVIEWS_PROJECT_ROOT", "/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews"))
# FEATURE_DIR: biến cấu hình/hằng số của notebook
FEATURE_DIR = PROJECT_ROOT / "artifacts" / "features"
# MODEL_DIR: biến cấu hình/hằng số của notebook
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
# ENSEMBLE_DIR: biến cấu hình/hằng số của notebook
ENSEMBLE_DIR = PROJECT_ROOT / "artifacts" / "ensemble"
# PREDICTION_DIR: biến cấu hình/hằng số của notebook
PREDICTION_DIR = PROJECT_ROOT / "artifacts" / "predictions"
# REPORT_TABLE_DIR: biến cấu hình/hằng số của notebook
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
# REPORT_FIGURE_DIR: biến cấu hình/hằng số của notebook
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
# PROCESSED_DIR: biến cấu hình/hằng số của notebook
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# for: vòng lặp — for directory in [MODEL_DIR, ENSEMBLE_DIR, PREDICTION_DIR, R
for directory in [MODEL_DIR, ENSEMBLE_DIR, PREDICTION_DIR, REPORT_TABLE_DIR, REPORT_FIGURE_DIR]:
    # directory.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
    directory.mkdir(parents=True, exist_ok=True)

# RAW_FEATURE_PATHS: biến cấu hình/hằng số của notebook
RAW_FEATURE_PATHS = {
    # "train": FEATURE_DIR / "features_raw_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "features_raw_train.npy",
    # "val": FEATURE_DIR / "features_raw_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "features_raw_val.npy",
    # "test": FEATURE_DIR / "features_raw_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "features_raw_test.npy",
# }: đóng khối từ điển
}
# LABEL_PATHS: biến cấu hình/hằng số của notebook
LABEL_PATHS = {
    # "train": FEATURE_DIR / "labels_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "labels_train.npy",
    # "val": FEATURE_DIR / "labels_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "labels_val.npy",
    # "test": FEATURE_DIR / "labels_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "labels_test.npy",
# }: đóng khối từ điển
}
# FEATURE_METADATA_PATH: biến cấu hình/hằng số của notebook
FEATURE_METADATA_PATH = FEATURE_DIR / "feature_metadata.json"

# random.seed(SEED): cố định seed random
random.seed(SEED)
# np.random.seed(SEED): cố định seed numpy
np.random.seed(SEED)


# utc_now: định nghĩa hàm utc now
def utc_now() -> str:
    # return: trả kết quả từ hàm
    return datetime.now(timezone.utc).isoformat()


# ensure_package: import hoặc pip install package
def ensure_package(import_name: str, pip_name: str | None = None):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)
    # except: xử lý ngoại lệ — except ImportError:
    except ImportError:
        # subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or impor...: thực thi lệnh Python
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)


# read_json: đọc file JSON
def read_json(path: Path, default=None):
    # if: điều kiện — if not path.exists():
    if not path.exists():
        # return: trả kết quả từ hàm
        return default if default is not None else {}
    # with: context manager — with path.open("r", encoding="utf-8") as file:
    with path.open("r", encoding="utf-8") as file:
        # return: parse nội dung JSON
        return json.load(file)


# load_raw_arrays: nạp feature/label .npy từ Phase 2
def load_raw_arrays():
    # missing = ...: kiểm tra file/thư mục tồn tại
    missing = [str(path) for path in list(RAW_FEATURE_PATHS.values()) + list(LABEL_PATHS.values()) if not path.exists()]
    # if: điều kiện — if missing:
    if missing:
        # raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(miss...: ghi dictionary ra JSON
        raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(missing, indent=2))
    # X: biến cấu hình/hằng số của notebook
    X = {split: np.load(path).astype(np.float32, copy=False) for split, path in RAW_FEATURE_PATHS.items()}
    # y = ...: nạp mảng từ file .npy
    y = {split: np.load(path).astype(np.int64, copy=False) for split, path in LABEL_PATHS.items()}
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # if: điều kiện — if X[split].ndim != 2:
        if X[split].ndim != 2:
            # raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}"): ném lỗi và dừng cell
            raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}")
        # if: điều kiện — if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
        if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
            # raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[spl...: ném lỗi và dừng cell
            raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[split].shape}")
        # if: điều kiện — if not np.isfinite(X[split]).all():
        if not np.isfinite(X[split]).all():
            # raise ValueError(f"Non-finite raw features in {split}"): ném lỗi và dừng cell
            raise ValueError(f"Non-finite raw features in {split}")
    # return: trả kết quả từ hàm
    return X, y


# safe_roc_auc: ROC-AUC an toàn
def safe_roc_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(roc_auc_score(y_true, prob_fake))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# safe_pr_auc: PR-AUC an toàn
def safe_pr_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(average_precision_score(y_true, prob_fake, pos_label=FAKE_LABEL))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# evaluate_predictions: tính metric classification từ xác suất
def evaluate_predictions(y_true, prob_fake, split, model_variant, threshold=DEFAULT_THRESHOLD, threshold_strategy="default_0.5"):
    # y_true = ...: ép kiểu dữ liệu cột
    y_true = np.asarray(y_true).astype(int)
    # prob_fake = ...: ép kiểu dữ liệu cột
    prob_fake = np.asarray(prob_fake).astype(float)
    # y_pred = ...: ép kiểu dữ liệu cột
    y_pred = (prob_fake >= threshold).astype(int)
    # precision, recall, f1, support = precision_recall_fscore_support(: thực thi lệnh Python
    precision, recall, f1, support = precision_recall_fscore_support(
        # y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0: thực thi lệnh Python
        y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0
    # ): đóng ngoặc gọi hàm
    )
    # tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL...: thực thi lệnh Python
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL]).ravel()
    # return: trả kết quả từ hàm
    return {
        # "generated_at_utc": utc_now(),: thực thi lệnh Python
        "generated_at_utc": utc_now(),
        # "seed": int(SEED),: ép kiểu số nguyên
        "seed": int(SEED),
        # "split": split,: thực thi lệnh Python
        "split": split,
        # "model_variant": model_variant,: thực thi lệnh Python
        "model_variant": model_variant,
        # "threshold": float(threshold),: ép kiểu số thực
        "threshold": float(threshold),
        # "threshold_strategy": threshold_strategy,: ép kiểu chuỗi
        "threshold_strategy": threshold_strategy,
        # "accuracy": float(accuracy_score(y_true, y_pred)),: ép kiểu số thực
        "accuracy": float(accuracy_score(y_true, y_pred)),
        # "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),: ép kiểu số thực
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        # "precision_fake": float(precision[1]),: ép kiểu số thực
        "precision_fake": float(precision[1]),
        # "recall_fake": float(recall[1]),: ép kiểu số thực
        "recall_fake": float(recall[1]),
        # "f1_fake": float(f1[1]),: ép kiểu số thực
        "f1_fake": float(f1[1]),
        # "support_real": int(support[0]),: ép kiểu số nguyên
        "support_real": int(support[0]),
        # "support_fake": int(support[1]),: ép kiểu số nguyên
        "support_fake": int(support[1]),
        # "roc_auc": safe_roc_auc(y_true, prob_fake),: thực thi lệnh Python
        "roc_auc": safe_roc_auc(y_true, prob_fake),
        # "pr_auc": safe_pr_auc(y_true, prob_fake),: thực thi lệnh Python
        "pr_auc": safe_pr_auc(y_true, prob_fake),
        # "brier_score": float(brier_score_loss(y_true, prob_fake)),: ép kiểu số thực
        "brier_score": float(brier_score_loss(y_true, prob_fake)),
        # "tn": int(tn),: ép kiểu số nguyên
        "tn": int(tn),
        # "fp": int(fp),: ép kiểu số nguyên
        "fp": int(fp),
        # "fn": int(fn),: ép kiểu số nguyên
        "fn": int(fn),
        # "tp": int(tp),: ép kiểu số nguyên
        "tp": int(tp),
    # }: đóng khối từ điển
    }


# save_probability: lưu xác suất fake ra file .npy
def save_probability(prob_fake, model_variant, split):
    # path = ...: gán giá trị cho biến path
    path = PREDICTION_DIR / f"{model_variant}_{split}_prob.npy"
    # np.save(path, np.asarray(prob_fake, dtype=np.float32)): lưu mảng numpy ra file .npy
    np.save(path, np.asarray(prob_fake, dtype=np.float32))
    # return: trả kết quả từ hàm
    return str(path)


# write_metrics: ghi bảng metric ra CSV
def write_metrics(prob_map, y, model_variant, output_name):
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y[split], prob_map[split], split=split, model_variant=model_variant)
        # row["probability_path"] = save_probability(prob_map[split], model_variant, split...: thực thi lệnh Python
        row["probability_path"] = save_probability(prob_map[split], model_variant, split)
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
    # df = ...: gán giá trị cho biến df
    df = pd.DataFrame(rows)
    # path = ...: gán giá trị cho biến path
    path = REPORT_TABLE_DIR / output_name
    # df.to_csv(path, index=False): ghi DataFrame ra file CSV
    df.to_csv(path, index=False)
    # display(df): hiển thị bảng/kết quả trên notebook
    display(df)
    # print("Saved metrics:", path): in thông tin ra console
    print("Saved metrics:", path)
    # return: trả kết quả từ hàm
    return df, path


In [4]:
# BASE_CANDIDATES: biến cấu hình/hằng số của notebook
BASE_CANDIDATES = {
    # "phase5_lgbm_raw": {split: PREDICTION_DIR / f"phase5_lgbm_raw_{split}_prob.npy" ...: thực thi lệnh Python
    "phase5_lgbm_raw": {split: PREDICTION_DIR / f"phase5_lgbm_raw_{split}_prob.npy" for split in ["train", "val", "test"]},
    # "phase5_xgb_raw": {split: PREDICTION_DIR / f"phase5_xgb_raw_{split}_prob.npy" fo...: thực thi lệnh Python
    "phase5_xgb_raw": {split: PREDICTION_DIR / f"phase5_xgb_raw_{split}_prob.npy" for split in ["train", "val", "test"]},
    # "phase5_mlp_raw": {split: PREDICTION_DIR / f"phase5_mlp_raw_{split}_prob.npy" fo...: thực thi lệnh Python
    "phase5_mlp_raw": {split: PREDICTION_DIR / f"phase5_mlp_raw_{split}_prob.npy" for split in ["train", "val", "test"]},
    # "phase5_cnn_bilstm_sequence": {split: PREDICTION_DIR / f"phase5_cnn_bilstm_seque...: thực thi lệnh Python
    "phase5_cnn_bilstm_sequence": {split: PREDICTION_DIR / f"phase5_cnn_bilstm_sequence_{split}_prob.npy" for split in ["train", "val", "test"]},
# }: đóng khối từ điển
}


# load_available_probability_maps: hàm xử lý load available probability maps
def load_available_probability_maps(candidate_paths=BASE_CANDIDATES):
    # available = ...: gán giá trị cho biến available
    available = {}
    # skipped = ...: gán giá trị cho biến skipped
    skipped = []
    # for: vòng lặp — for candidate, split_paths in candidate_paths.items():
    for candidate, split_paths in candidate_paths.items():
        # if: điều kiện — if all(path.exists() for path in split_paths.values()):
        if all(path.exists() for path in split_paths.values()):
            # available[candidate] = {split: np.load(path).astype(np.float32) for split, path ...: nạp mảng từ file .npy
            available[candidate] = {split: np.load(path).astype(np.float32) for split, path in split_paths.items()}
        # else: nhánh còn lại của điều kiện
        else:
            # skipped.append(candidate): thực thi lệnh Python
            skipped.append(candidate)
    # if: điều kiện — if not available:
    if not available:
        # raise FileNotFoundError("No complete candidate probability sets found. Run base ...: ném lỗi và dừng cell
        raise FileNotFoundError("No complete candidate probability sets found. Run base model notebooks first.")
    # print("Available candidates:", sorted(available)): sắp xếp danh sách
    print("Available candidates:", sorted(available))
    # if: điều kiện — if skipped:
    if skipped:
        # print("Skipped incomplete candidates:", skipped): in thông tin ra console
        print("Skipped incomplete candidates:", skipped)
    # return: trả kết quả từ hàm
    return available


# threshold_sweep: hàm xử lý threshold sweep
def threshold_sweep(y_true, prob_fake, model_variant, split="val"):
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for threshold in np.round(np.arange(0.01, 1.00, 0.01), 2):
    for threshold in np.round(np.arange(0.01, 1.00, 0.01), 2):
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y_true, prob_fake, split, model_variant, threshold=float(threshold), threshold_strategy="threshold_sweep")
        # row["target_precision_fake"] = TARGET_PRECISION_FAKE: thực thi lệnh Python
        row["target_precision_fake"] = TARGET_PRECISION_FAKE
        # row["target_met"] = bool(row["precision_fake"] >= TARGET_PRECISION_FAKE): ép kiểu boolean
        row["target_met"] = bool(row["precision_fake"] >= TARGET_PRECISION_FAKE)
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
    # return: trả kết quả từ hàm
    return pd.DataFrame(rows)


In [5]:
# from sklearn.calibration import CalibratedClassifierCV: thư viện machine learning scikit-learn
from sklearn.calibration import CalibratedClassifierCV
# from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier: thư viện machine learning scikit-learn
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
# from sklearn.linear_model import LogisticRegression: thư viện machine learning scikit-learn
from sklearn.linear_model import LogisticRegression

# _, y = load_raw_arrays(): thực thi lệnh Python
_, y = load_raw_arrays()
# base_probs = ...: gán giá trị cho biến base probs
base_probs = load_available_probability_maps()
# ORDER: biến cấu hình/hằng số của notebook
ORDER = sorted(base_probs)

# stack_matrix: hàm xử lý stack matrix
def stack_matrix(split):
    # return: ép kiểu dữ liệu cột
    return np.column_stack([base_probs[name][split] for name in ORDER]).astype(np.float32)

# X_stack = ...: gán giá trị cho biến X stack
X_stack = {split: stack_matrix(split) for split in ["train", "val", "test"]}
# STACKERS: biến cấu hình/hằng số của notebook
STACKERS = {
    # "phase5_stack_logistic": LogisticRegression(max_iter=2000, class_weight="balance...: lấy giá trị lớn nhất
    "phase5_stack_logistic": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
    # "phase5_stack_random_forest": RandomForestClassifier(n_estimators=300, max_depth...: lấy giá trị nhỏ nhất
    "phase5_stack_random_forest": RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=2, class_weight="balanced", random_state=SEED, n_jobs=-1),
    # "phase5_stack_extra_trees": ExtraTreesClassifier(n_estimators=300, max_depth=8, ...: lấy giá trị nhỏ nhất
    "phase5_stack_extra_trees": ExtraTreesClassifier(n_estimators=300, max_depth=8, min_samples_leaf=2, class_weight="balanced", random_state=SEED, n_jobs=-1),
# }: đóng khối từ điển
}
# candidate_probs = ...: gán giá trị cho biến candidate probs
candidate_probs = {}
# for: vòng lặp — for name, model in STACKERS.items():
for name, model in STACKERS.items():
    # model.fit(X_stack["train"], y["train"]): fit model/reducer trên dữ liệu train
    model.fit(X_stack["train"], y["train"])
    # candidate_probs[name] = {split: model.predict_proba(X_stack[split])[:, FAKE_LABE...: dự đoán nhãn/xác suất
    candidate_probs[name] = {split: model.predict_proba(X_stack[split])[:, FAKE_LABEL].astype(np.float32) for split in ["train", "val", "test"]}
    # joblib.dump({"model": model, "candidate_order": ORDER, "seed": SEED}, ENSEMBLE_D...: lưu object (scaler/model) ra disk
    joblib.dump({"model": model, "candidate_order": ORDER, "seed": SEED}, ENSEMBLE_DIR / f"{name}.joblib")

# cal_name = ...: gán giá trị cho biến cal name
cal_name = "phase5_stack_logistic_sigmoid_calibrated"
# calibrator = ...: lấy giá trị lớn nhất
calibrator = CalibratedClassifierCV(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED), cv=5, method="sigmoid")
# calibrator.fit(X_stack["train"], y["train"]): fit model/reducer trên dữ liệu train
calibrator.fit(X_stack["train"], y["train"])
# candidate_probs[cal_name] = {split: calibrator.predict_proba(X_stack[split])[:, ...: dự đoán nhãn/xác suất
candidate_probs[cal_name] = {split: calibrator.predict_proba(X_stack[split])[:, FAKE_LABEL].astype(np.float32) for split in ["train", "val", "test"]}
# joblib.dump({"model": calibrator, "candidate_order": ORDER, "seed": SEED}, ENSEM...: lưu object (scaler/model) ra disk
joblib.dump({"model": calibrator, "candidate_order": ORDER, "seed": SEED}, ENSEMBLE_DIR / f"{cal_name}.joblib")

# rows = ...: gán giá trị cho biến rows
rows = []
# for: vòng lặp — for name, prob_map in candidate_probs.items():
for name, prob_map in candidate_probs.items():
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y[split], prob_map[split], split, name)
        # row["candidate_type"] = "stacking_calibration": thực thi lệnh Python
        row["candidate_type"] = "stacking_calibration"
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
# metrics_df = ...: gán giá trị cho biến metrics df
metrics_df = pd.DataFrame(rows)
# metrics_path = ...: gán giá trị cho biến metrics path
metrics_path = REPORT_TABLE_DIR / "phase5_stacking_calibration_metrics.csv"
# metrics_df.to_csv(metrics_path, index=False): ghi DataFrame ra file CSV
metrics_df.to_csv(metrics_path, index=False)
# selected_name = ...: gán giá trị cho biến selected name
selected_name = metrics_df[metrics_df["split"].eq("val")].sort_values(["macro_f1", "roc_auc", "precision_fake", "recall_fake"], ascending=[False, False, False, False]).iloc[0]["model_variant"]
# selected_prob_map = ...: gán giá trị cho biến selected prob map
selected_prob_map = candidate_probs[selected_name]
# selected_metrics_df, selected_metrics_path = write_metrics(selected_prob_map, y,...: thực thi lệnh Python
selected_metrics_df, selected_metrics_path = write_metrics(selected_prob_map, y, "phase5_stacking_calibrated", "phase5_stacking_calibrated_metrics.csv")
# with: context manager — with (ENSEMBLE_DIR / "phase5_stacking_calibrated_metadata.js
with (ENSEMBLE_DIR / "phase5_stacking_calibrated_metadata.json").open("w", encoding="utf-8") as file:
    # json.dump({"candidate_order": ORDER, "selected_name": selected_name, "metrics_pa...: ghi dictionary ra JSON
    json.dump({"candidate_order": ORDER, "selected_name": selected_name, "metrics_path": str(metrics_path), "selected_metrics_path": str(selected_metrics_path)}, file, indent=2)
# display(metrics_df[metrics_df["split"].eq("val")].sort_values("macro_f1", ascend...: hiển thị bảng/kết quả trên notebook
display(metrics_df[metrics_df["split"].eq("val")].sort_values("macro_f1", ascending=False))


Available candidates: ['phase5_cnn_bilstm_sequence', 'phase5_lgbm_raw', 'phase5_mlp_raw', 'phase5_xgb_raw']


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,support_real,support_fake,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,probability_path
0,2026-06-09T19:33:31.199119+00:00,42,train,phase5_stacking_calibrated,0.5,default_0.5,0.999967,0.999965,0.999918,1.000000,...,17677,12246,1.000000,1.000000,0.000044,17676,1,0,12246,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
1,2026-06-09T19:33:31.239101+00:00,42,val,phase5_stacking_calibrated,0.5,default_0.5,0.913301,0.907569,0.971715,0.811738,...,3789,2624,0.972420,0.970699,0.077922,3727,62,494,2130,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
2,2026-06-09T19:33:31.292596+00:00,42,test,phase5_stacking_calibrated,0.5,default_0.5,0.915952,0.910491,0.972789,0.817454,...,3789,2624,0.973124,0.971523,0.074717,3729,60,479,2145,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...


Saved metrics: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/phase5_stacking_calibrated_metrics.csv


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,support_real,support_fake,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,candidate_type
1,2026-06-09T19:33:30.428467+00:00,42,val,phase5_stack_logistic,0.5,default_0.5,0.913301,0.907569,0.971715,0.811738,...,3789,2624,0.972420,0.970699,0.077922,3727,62,494,2130,stacking_calibration
10,2026-06-09T19:33:30.565288+00:00,42,val,phase5_stack_logistic_sigmoid_calibrated,0.5,default_0.5,0.913145,0.907349,0.972998,0.810213,...,3789,2624,0.972628,0.970922,0.080155,3730,59,498,2126,stacking_calibration
7,2026-06-09T19:33:30.525463+00:00,42,val,phase5_stack_extra_trees,0.5,default_0.5,0.909091,0.902959,0.968764,0.803735,...,3789,2624,0.966355,0.957197,0.081104,3721,68,515,2109,stacking_calibration
4,2026-06-09T19:33:30.485249+00:00,42,val,phase5_stack_random_forest,0.5,default_0.5,0.908311,0.902037,0.969991,0.800686,...,3789,2624,0.964440,0.953047,0.087142,3724,65,523,2101,stacking_calibration
